In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
patients_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.patients")

display(patients_df)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,ingestion_timestamp,source_file,load_date
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02


In [0]:
patients_df.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- contact_number: long (nullable = true)
 |-- address: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- insurance_number: string (nullable = true)
 |-- email: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_date: date (nullable = true)



In [0]:
print("Total Records :", patients_df.count())

Total Records : 50


In [0]:
patients_df.groupBy("patient_id") \
           .count() \
           .filter(col("count") > 1) \
           .show()

+----------+-----+
|patient_id|count|
+----------+-----+
+----------+-----+



In [0]:
patients_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in patients_df.columns
]).show()

+----------+----------+---------+------+-------------+--------------+-------+-----------------+------------------+----------------+-----+-------------------+-----------+---------+
|patient_id|first_name|last_name|gender|date_of_birth|contact_number|address|registration_date|insurance_provider|insurance_number|email|ingestion_timestamp|source_file|load_date|
+----------+----------+---------+------+-------------+--------------+-------+-----------------+------------------+----------------+-----+-------------------+-----------+---------+
|         0|         0|        0|     0|            0|             0|      0|                0|                 0|               0|    0|                  0|          0|        0|
+----------+----------+---------+------+-------------+--------------+-------+-----------------+------------------+----------------+-----+-------------------+-----------+---------+



In [0]:
silver_patients = (
    patients_df

    .dropDuplicates(["patient_id"])

    .withColumn(
        "first_name",
        initcap(trim(col("first_name")))
    )

    .withColumn(
        "last_name",
        initcap(trim(col("last_name")))
    )

    .withColumn(
        "gender",
        upper(trim(col("gender")))
    )

    .withColumn(
        "email",
        lower(trim(col("email")))
    )
)

In [0]:
silver_patients = silver_patients.withColumn(
    "age",
    floor(
        months_between(
            current_date(),
            col("date_of_birth")
        ) / 12
    )
)

In [0]:
silver_patients = silver_patients.filter(col("age") >= 0)

In [0]:
silver_patients = silver_patients.filter(
    col("insurance_number").isNotNull()
)

In [0]:
silver_patients = (
    silver_patients

    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
)

In [0]:
(
    silver_patients.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.patients")
)

In [0]:
display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.patients")
)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,ingestion_timestamp,source_file,load_date,age,silver_load_timestamp
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,45,2026-07-02T06:59:05.974Z
P011,Emily,Jones,F,1966-12-04,8990604070,789 Pine Rd,2022-09-27,MedCare Plus,INS172991,emily.jones@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,59,2026-07-02T06:59:05.974Z
P014,Alex,Taylor,M,1968-02-27,7292262512,789 Pine Rd,2023-12-12,MedCare Plus,INS118070,alex.taylor@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,58,2026-07-02T06:59:05.974Z
P045,Linda,Miller,F,1966-04-25,7579616535,321 Maple Dr,2021-01-23,MedCare Plus,INS701863,linda.miller@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,60,2026-07-02T06:59:05.974Z
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,54,2026-07-02T06:59:05.974Z
P018,Laura,Wilson,M,1979-09-24,7145815738,789 Pine Rd,2022-09-23,PulseSecure,INS635017,laura.wilson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,46,2026-07-02T06:59:05.974Z
P025,Robert,Wilson,M,1966-08-14,7482069727,123 Elm St,2021-09-09,HealthIndia,INS833429,robert.wilson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,59,2026-07-02T06:59:05.974Z
P039,Jane,Wilson,F,1950-12-12,9271131338,789 Pine Rd,2021-03-09,PulseSecure,INS348710,jane.wilson@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,75,2026-07-02T06:59:05.974Z
P028,Alex,Moore,M,1993-04-13,7028910482,321 Maple Dr,2023-05-20,MedCare Plus,INS679036,alex.moore@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,33,2026-07-02T06:59:05.974Z
P038,David,Smith,M,1991-06-25,6347262390,789 Pine Rd,2021-04-19,MedCare Plus,INS580761,david.smith@mail.com,2026-07-02T06:36:35.908Z,patients.csv,2026-07-02,35,2026-07-02T06:59:05.974Z
